## Validating pre-trained models
This code demonstrates how to load our trained models and one way of doing using them for inference.
It requires that you downloaded and extracted the [pretrained models and the corresponding preprocessed version of kinodata-3D](https://zenodo.org/records/10410594)
in the root directory of this repository.

In [ ]:
import json
from pathlib import Path
from typing import Any

import kinodata.configuration as cfg
from kinodata.model import RegressionModel
from kinodata.model.complex_transformer import make_model as make_complex_transformer
from kinodata.model.dti import make_model as make_dti_baseline
from kinodata.data.data_module import make_kinodata_module
from kinodata.transform import TransformToComplexGraph

In [2]:
!wandb disabled

W&B disabled.


Demo boilerplate code for loading model checkpoints, reuses parts of our training/evaluation code.

In [3]:
model_dir = Path("..") / "models"
assert model_dir.exists()

def path_to_model(rmsd_threshold: int, split_type: str, split_fold: int, model_type: str) -> Path:
    p = model_dir / f"rmsd_cutoff_{rmsd_threshold}" / split_type / str(split_fold) / model_type
    if not p.exists():
        p.mkdir(parents=True)
    return p
model_cls = {
    "DTI": make_dti_baseline,
    "CGNN": make_complex_transformer,
    "CGNN-3D": make_complex_transformer
}

def load_wandb_config(
    config_file: Path
) -> dict[str, Any]:
    with open(config_file, "r") as f_config:
        config = json.load(f_config)
    config = {str(key): value["value"] for key, value in config.items()}
    return config

def load_from_checkpoint(
    rmsd_threshold: int,
    split_type: str,
    fold: int,
    model_type: str
) -> RegressionModel:
    cls = model_cls[model_type]
    p = path_to_model(rmsd_threshold, split_type, fold, model_type)
    model_ckpt = list(p.glob("**/*.ckpt"))[0]
    model_config = p / "config.json"
    ckp = torch.load(model_ckpt, map_location="cpu")
    config = cfg.Config(load_wandb_config(model_config))
    model = cls(config)
    assert isinstance(model, RegressionModel)
    model.load_state_dict(ckp["state_dict"])
    return model

Load model checkpoints for *scaffold-split* data subject to predicted RMSD $\leq 4\text{Å}$, where the $0$-th fold is used as test set.

In [4]:
cgnn = load_from_checkpoint(4, "scaffold-k-fold", 0, "CGNN")
cgnn_3d = load_from_checkpoint(4, "scaffold-k-fold", 0, "CGNN-3D") 
dti = load_from_checkpoint(4, "scaffold-k-fold", 0, "DTI")

/var/folders/gp/_kdyh3hn1yv47p6w56krv9000000gn/T/ipykernel_59903/1392587981.py:33: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckp = torch.load(model_ckpt, map_location="c

Create the matching data module

In [5]:
data_module = make_kinodata_module(
    cfg.get("data", "training").update(
        dict(
            batch_size=32,
            split_type="scaffold-k-fold",
            filter_rmsd_max_value=4.0,
            split_index=0,
        )
    ),
    transforms=[TransformToComplexGraph(remove_heterogeneous_representation=False)],
)

/Users/joschkagross/projects/release_working_kinodata3d/kinodata-3D-affinity-prediction/kinodata/data/dataset.py:346: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  if osp.ex

Creating data module:
    split:Split[int](train=46640, val=5830, test=5830, source=/Users/joschkagross/projects/release_working_kinodata3d/kinodata-3D-affinity-prediction/data/processed/filter_predicted_rmsd_le4.00/scaffold-k-fold/1:5.csv)
    train_transform:Compose([
  TransformToComplexGraph()
])
    val_transform:Compose([
  TransformToComplexGraph()
])


/Users/joschkagross/projects/release_working_kinodata3d/kinodata-3D-affinity-prediction/kinodata/data/dataset.py:346: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  if osp.ex

Fast way of demonstrating inference on just one test batch:

In [6]:
demo_test_batch = next(iter(data_module.test_dataloader()))
with torch.no_grad():
    mae_sample = {
        "DTI sample test MAE": dti.test_step(demo_test_batch)["test/mae"],
        "CGNN sample test MAE": cgnn.test_step(demo_test_batch)["test/mae"],
        "CGNN-3D sample test MAE": cgnn_3d.test_step(demo_test_batch)["test/mae"],
    }
mae_sample

/opt/homebrew/Caskroom/miniforge/base/envs/kinodata/lib/python3.10/site-packages/pytorch_lightning/core/module.py:377: UserWarning: You are trying to `self.log()` but the `self.trainer` reference is not registered on the model yet. This is most likely because the model hasn't been passed to the `Trainer`
  rank_zero_warn(


{'DTI sample test MAE': tensor(0.6814),
 'CGNN sample test MAE': tensor(0.6193),
 'CGNN-3D sample test MAE': tensor(0.5502)}

Test all three models using all test data in the current data module

In [7]:
from pytorch_lightning import Trainer

In [8]:
trainer = Trainer(logger=False)
dti_metrics = trainer.test(model=dti, datamodule=data_module, ckpt_path=None)
cgnn_metrics = trainer.test(model=cgnn, datamodule=data_module, ckpt_path=None)
cgnn_3d_metrics = trainer.test(model=cgnn_3d, datamodule=data_module, ckpt_path=None)

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
/opt/homebrew/Caskroom/miniforge/base/envs/kinodata/lib/python3.10/site-packages/pytorch_lightning/trainer/setup.py:200: UserWarning: MPS available but not used. Set `accelerator` and `devices` using `Trainer(accelerator='mps', devices=1)`.
  rank_zero_warn(
/opt/homebrew/Caskroom/miniforge/base/envs/kinodata/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:224: PossibleUserWarning: The dataloader, test_dataloader 0, does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` (try 12 which is the number of cpus on this machine) in the `DataLoader` init to improve performance.
  rank_zero_warn(


Testing DataLoader 0: 100%|██████████| 183/183 [00:08<00:00, 22.16it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test/corr           0.6359230875968933
        test/mae            0.7256727814674377
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Testing DataLoader 0: 100%|██████████| 183/183 [01:19<00:00,  2.31it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test/corr           0.6851842999458313
        test/mae      

Display results

In [ ]:
{
    "DTI": dti_metrics, 
    "CGNN": cgnn_metrics, 
    "CGNN-3D": cgnn_3d_metrics
}

{'DTI': [{'test/mae': 0.8224371671676636, 'test/corr': 0.5125585198402405}],
 'CGNN': [{'test/mae': 0.8519962430000305, 'test/corr': 0.4546387493610382}],
 'CGNN-3D': [{'test/mae': 0.7448561787605286, 'test/corr': 0.632169246673584}]}